# 01_eda.ipynb - EDA + Backtest Generalizado

**Pipeline**: Carga universo admitido → Features → Señales entrada → Backtest con Csl calibrado → Trades + Métricas

Configuración centralizada en `config/settings.py`

## 1. Imports y Configuración

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Configuración centralizada
from config.settings import DATA, FEATURES, RISK, BACKTEST, SIGNALS, get_universe_tickers

# Módulos core
from src.data import load_ohlc_from_yfinance, validate_ohlc_data
from src.features import add_all_features, calculate_efficiency_ratio, filter_momentum_entries
from src.risk import run_backtest_loop, calculate_performance_metrics

print(f"Config cargada: {DATA.source} {DATA.interval} {DATA.period}")
print(f"Capital: ${BACKTEST.capital:,.0f}, Max bars: {BACKTEST.max_bars}, TP/SL: {BACKTEST.tp_sl_ratio}:1")
print(f"Risk/trade: ${RISK.risk_per_trade:.0f}")

Config cargada: yfinance 1h 730d
Capital: $10,000, Max bars: 40, TP/SL: 1.5:1
Risk/trade: $100


## 2. Cargar Universo Admitido (Csl + BP calibrados)

In [2]:
admitted_path = REPO_ROOT / "data" / "universe_admitted.csv"
admitted_df = pd.read_csv(admitted_path)
print(f"Activos admitidos: {len(admitted_df)}")
admitted_df

Activos admitidos: 9


,ticker,csl,bp_median,bp_max,n_samples
0,MCRB,2.045802,1662.081848,2817.720239,500
1,LRMR,2.237089,1702.988206,2821.426409,500
2,ABSI,2.306791,1556.522468,2728.727235,500
3,IMUX,2.310284,1591.100857,2793.183932,500
4,HBIO,2.415738,1477.663359,2647.546162,500
5,KPTI,2.447570,1288.447781,2487.226090,500
6,HOTH,2.462243,1447.888784,2581.946600,500
7,SGMO,2.540506,928.517813,1735.888970,500
8,EBON,2.719857,1191.428107,1953.351230,500


## 3. Cargar Datos + Features + Señales + Backtest por Activo

In [3]:
all_trades = []
all_metrics = []

for _, row in admitted_df.iterrows():
    ticker = row['ticker']
    csl = row['csl']
    
    print(f"\n=== {ticker} (Csl={csl:.3f}) ===")
    
    try:
        # 1. Cargar datos OHLC
        df = load_ohlc_from_yfinance(ticker, period=DATA.period, interval=DATA.interval)
        
        if not validate_ohlc_data(df, min_rows=FEATURES.atr_period + RISK.lookforward_window + 100):
            print(f"  ❌ Datos insuficientes: {len(df)} barras")
            continue
        
        # 2. Generar todas las features (incluye ATR, ER, estructura, velas)
        df = add_all_features(
            df,
            atr_period=FEATURES.atr_period,
            sma_period=FEATURES.sma_period,
            roc_period=FEATURES.roc_period
        )
        
        # 3. Señal de entrada: 4 condiciones parametrizables
        signal = pd.Series(False, index=df.index)
        
        cond1 = pd.Series(True, index=df.index)
        if SIGNALS.use_er_filter:
            er = calculate_efficiency_ratio(df, k=FEATURES.er_k)
            cond1 = er > FEATURES.er_threshold
        
        cond2 = pd.Series(True, index=df.index)
        if SIGNALS.use_sma_trend:
            cond2 = df['Close'] > df['SMA']
        
        cond3 = pd.Series(True, index=df.index)
        if SIGNALS.use_structure_break:
            cond3 = df['Estructura_OK'] == True
        
        cond4 = pd.Series(True, index=df.index)
        if SIGNALS.use_volume_confirm:
            vol_ma = df['Volume'].rolling(FEATURES.vol_ma_period).mean()
            cond4 = df['Volume'] > FEATURES.vol_multiplier * vol_ma
        
        signal = cond1 & cond2 & cond3 & cond4
        
        n_signals = signal.sum()
        print(f"  Señales generadas: {n_signals}")
        
        if n_signals == 0:
            print(f"  ⚠️ Sin señales, saltando")
            continue
        
        # 4. Backtest con Csl calibrado del activo
        trades = run_backtest_loop(
            df=df,
            entry_signal=signal,
            c_sl=csl,
            c_tp=None,  # usa ratio 1.5 del config
            max_bars=BACKTEST.max_bars,
            risk_per_trade=RISK.risk_per_trade,
            capital=BACKTEST.capital
        )
        
        if len(trades) == 0:
            print(f"  ⚠️ Sin trades ejecutados")
            continue
        
        trades['ticker'] = ticker
        all_trades.append(trades)
        
        # 5. Métricas
        metrics = calculate_performance_metrics(trades, BACKTEST.capital)
        metrics['ticker'] = ticker
        metrics['csl'] = csl
        metrics['n_signals'] = n_signals
        metrics['n_trades'] = len(trades)
        all_metrics.append(metrics)
        
        print(f"  ✅ Trades: {len(trades)}, WinRate: {metrics.get('Win Rate', 0):.2%}, PF: {metrics.get('Profit Factor', 0):.2f}")
        
    except Exception as e:
        print(f"  ❌ Error {ticker}: {e}")
        continue


=== MCRB (Csl=2.046) ===


  Señales generadas: 105
  ✅ Trades: 68, WinRate: 33.82%, PF: 0.74

=== LRMR (Csl=2.237) ===


  Señales generadas: 107
  ✅ Trades: 66, WinRate: 39.39%, PF: 0.96

=== ABSI (Csl=2.307) ===


  Señales generadas: 111
  ✅ Trades: 77, WinRate: 44.16%, PF: 1.21

=== IMUX (Csl=2.310) ===


  Señales generadas: 101
  ✅ Trades: 68, WinRate: 38.24%, PF: 0.93

=== HBIO (Csl=2.416) ===


  Señales generadas: 78
  ✅ Trades: 60, WinRate: 35.00%, PF: 0.76

=== KPTI (Csl=2.448) ===


  Señales generadas: 85
  ✅ Trades: 59, WinRate: 30.51%, PF: 0.63

=== HOTH (Csl=2.462) ===


  Señales generadas: 60
  ✅ Trades: 45, WinRate: 40.00%, PF: 0.87

=== SGMO (Csl=2.541) ===


  Señales generadas: 90
  ✅ Trades: 66, WinRate: 34.85%, PF: 0.72

=== EBON (Csl=2.720) ===


  Señales generadas: 59
  ✅ Trades: 35, WinRate: 42.86%, PF: 1.16


## 4. Consolidar Resultados

In [4]:
if all_trades:
    trades_all = pd.concat(all_trades, ignore_index=True)
    metrics_all = pd.DataFrame(all_metrics)
    
    # Guardar
    out_dir = REPO_ROOT / "data"
    trades_all.to_csv(out_dir / "trades_all.csv", index=False)
    metrics_all.to_csv(out_dir / "metrics_all.csv", index=False)
    
    print(f"\n=== RESUMEN GLOBAL ===")
    print(f"Total trades: {len(trades_all)}")
    print(f"Activos con trades: {trades_all['ticker'].nunique()}")
    print(f"\nMétricas por activo:")
    print(metrics_all[['ticker', 'csl', 'n_trades', 'Win Rate', 'Profit Factor', 'Expectancy (USD)', 'Max Drawdown (%)', 'Sharpe Ratio', 'Retorno Total (%)']].to_string(index=False))
    
    # Métricas agregadas (portfolio simple equal weight)
    total_pnl = trades_all['PnL'].sum()
    total_ret = total_pnl / BACKTEST.capital * 100
    print(f"\n=== PORTFOLIO AGREGADO (equal weight) ===")
    print(f"PnL Total: ${total_pnl:,.2f}")
    print(f"Retorno Total: {total_ret:.2f}%")
    
    trades_all.head(10)
else:
    print("❌ No se generaron trades en ningún activo")


=== RESUMEN GLOBAL ===
Total trades: 544
Activos con trades: 9

Métricas por activo:
ticker      csl  n_trades  Win Rate  Profit Factor  Expectancy (USD)  Max Drawdown (%)  Sharpe Ratio  Retorno Total (%)
  MCRB 2.045802        68  0.338235       0.737345        -16.263155        -16.068893     -2.298967         -11.058945
  LRMR 2.237089        66  0.393939       0.962234         -2.204938        -10.846491     -0.293687          -1.455259
  ABSI 2.306791        77  0.441558       1.206234         10.805661         -5.027371      1.417139           8.320359
  IMUX 2.310284        68  0.382353       0.930634         -4.238119         -8.208928     -0.554575          -2.881921
  HBIO 2.415738        60  0.350000       0.757032        -14.875456        -12.066521     -2.092182          -8.925274
  KPTI 2.447570        59  0.305085       0.634094        -24.499548        -22.019218     -3.489209         -14.454733
  HOTH 2.462243        45  0.400000       0.866639         -7.998154      

## 5. Análisis Básico de Calidad (Preview para 03_backtest_evaluation.ipynb)

In [5]:
if all_trades:
    # Distribución motivos de salida
    print("=== MOTIVOS DE SALIDA ===")
    print(trades_all['Motivo'].value_counts())
    
    # MAE / MFE promedio
    print(f"\n=== MAE/MFE PROMEDIO ===")
    print(f"MAE medio: ${trades_all['MAE'].mean():.2f}")
    print(f"MFE medio: ${trades_all['MFE'].mean():.2f}")
    print(f"Ratio MFE/MAE: {trades_all['MFE'].mean() / trades_all['MAE'].mean():.2f}")
    
    # PnL por motivo
    print(f"\n=== PnL POR MOTIVO ===")
    print(trades_all.groupby('Motivo')['PnL'].agg(['count', 'mean', 'sum']))
    
    # Duración media
    print(f"\nDuración media: {trades_all['Velas'].mean():.1f} barras")
    print(f"Max duración: {trades_all['Velas'].max()} barras")

=== MOTIVOS DE SALIDA ===
Motivo
SL           319
TP           183
Time Exit     42
Name: count, dtype: int64

=== MAE/MFE PROMEDIO ===
MAE medio: $0.57
MFE medio: $0.57
Ratio MFE/MAE: 1.01

=== PnL POR MOTIVO ===
           count        mean           sum
Motivo                                    
SL           319  -99.723627 -31811.836942
TP           183  149.616442  27379.808884
Time Exit     42    5.042030    211.765246

Duración media: 12.2 barras
Max duración: 40 barras
